### Using OpenAI (GPT-5.4) for data annotation

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [3]:
from openai import OpenAI 
import json
from util.preprocessing import parse_iob2_file

#### Toy experiment with annotation for one sentence

In [1]:
one_shot = """
1	350	O
2	,	O
3	Wellesley	B-LOC
4	,	O
5	Massachusetts	B-LOC
6	02481	O
7	doing	O
8	business	O
9	as	O
10	"	O
11	Silicon	B-LOC
12	Valley	I-LOC
13	East	I-LOC
14	"	O
15	and	O
16	AKAMAI	B-ORG
17	TECHNOLOGIES	I-ORG
18	,	O
19	INC	O
20	.	O
21	("	O
22	Borrower	B-PER
23	"),	O
"""

one_shot = one_shot.strip().split('\n')

example_input_lines = []
example_output_lines = []

for line in one_shot:
    _, token, label = line.strip().split('\t')
    example_input_lines.append(token)
    example_output_lines.append(f"{token}\t{label}")

example_input = '\n'.join(example_input_lines)
example_output = '\n'.join(example_output_lines)


test_case = """60	referred	O
61	to	O
62	as	O
63	"	O
64	Lender	B-PER
65	"),	O
66	and	O
67	Acme	B-ORG
68	Inc	I-ORG
69	.,	MISMATCH
70	a	MISMATCH
71	corporation	MISMATCH
72	organized	MISMATCH
73	and	MISMATCH
74	existing	MISMATCH
75	under	MISMATCH
76	the	MISMATCH
77	laws	MISMATCH
78	of	MISMATCH
79	the	MISMATCH
80	United	MISMATCH
81	States	MISMATCH
82	of	MISMATCH
83	America	MISMATCH
84	,	MISMATCH
85	with	MISMATCH
86	its	MISMATCH
87	principal	MISMATCH
88	place	MISMATCH
89	of	MISMATCH
90	business	MISMATCH
91	at	MISMATCH
92	123	MISMATCH
93	Main	MISMATCH
94	Street	MISMATCH
95	,	MISMATCH
96	Anytown	GOLD_SKIPPED
97	,	O
98	USA	MISMATCH
99	12345	MISMATCH
100	(	MISMATCH
101	hereinafter	MISMATCH
102	referred	MISMATCH
103	to	MISMATCH
104	as	MISMATCH
105	"	MISMATCH
106	Borrower	MISMATCH
107	").	MISMATCH"""


test_case = test_case.strip().split('\n')

test_input_lines = []
test_output_lines = []

for line in test_case:
    _, token, label = line.strip().split('\t')
    test_input_lines.append(token)
    test_output_lines.append(f"{token}\t{label}")

test_input = '\n'.join(test_input_lines)
test_output = '\n'.join(test_output_lines)

print(example_input)
print(example_output)
print(test_input)
print(test_output)

350
,
Wellesley
,
Massachusetts
02481
doing
business
as
"
Silicon
Valley
East
"
and
AKAMAI
TECHNOLOGIES
,
INC
.
("
Borrower
"),
350	O
,	O
Wellesley	B-LOC
,	O
Massachusetts	B-LOC
02481	O
doing	O
business	O
as	O
"	O
Silicon	B-LOC
Valley	I-LOC
East	I-LOC
"	O
and	O
AKAMAI	B-ORG
TECHNOLOGIES	I-ORG
,	O
INC	O
.	O
("	O
Borrower	B-PER
"),	O
referred
to
as
"
Lender
"),
and
Acme
Inc
.,
a
corporation
organized
and
existing
under
the
laws
of
the
United
States
of
America
,
with
its
principal
place
of
business
at
123
Main
Street
,
Anytown
,
USA
12345
(
hereinafter
referred
to
as
"
Borrower
").
referred	O
to	O
as	O
"	O
Lender	B-PER
"),	O
and	O
Acme	B-ORG
Inc	I-ORG
.,	MISMATCH
a	MISMATCH
corporation	MISMATCH
organized	MISMATCH
and	MISMATCH
existing	MISMATCH
under	MISMATCH
the	MISMATCH
laws	MISMATCH
of	MISMATCH
the	MISMATCH
United	MISMATCH
States	MISMATCH
of	MISMATCH
America	MISMATCH
,	MISMATCH
with	MISMATCH
its	MISMATCH
principal	MISMATCH
place	MISMATCH
of	MISMATCH
business	MISMATCH
at	MISMATCH
123	MIS

In [2]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-5.4",
    input=(

    "You are a strict NER tagger.\n\n"

    "Task:\n"
    "Assign a BIO tag to EACH token.\n\n"

    "Allowed labels:\n"
    "B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC, O\n\n"

    "Rules:\n"
    "- Annotate legal party roles \"Lender\" and \"Borrower\" as B-PER\n"
    "- EXACTLY one label per token\n"
    "- SAME number of output lines as input tokens\n"
    "- SAME token order as input\n"
    "- Do not modify tokens\n"
    "- Output format must be: token<TAB>label\n"
    "- One token-label pair per line\n"
    "- No explanations\n\n"

    "Example input:\n"
    f"{example_input}\n\n"

    "Example output:\n"
    f"{example_output}\n\n"

    "Now annotate this input:\n"
    f"{test_input}"
    )
)

openai_response = response.output[0].content[0].text
print(openai_response)

referred	O
to	O
as	O
"	O
Lender	B-PER
"),	O
and	O
Acme	B-ORG
Inc	B-ORG
.,	O
a	O
corporation	O
organized	O
and	O
existing	O
under	O
the	O
laws	O
of	O
the	O
United	B-LOC
States	I-LOC
of	I-LOC
America	I-LOC
,	O
with	O
its	O
principal	O
place	O
of	O
business	O
at	O
123	O
Main	O
Street	O
,	O
Anytown	B-LOC
,	O
USA	B-LOC
12345	O
(	O
hereinafter	O
referred	O
to	O
as	O
"	O
Borrower	B-PER
").	O


In [3]:
openai_labels = openai_response.strip().split('\n')
true_labels = test_output.strip().split('\n')

assert len(openai_labels) == len(true_labels), "Number of labels does not match number of tokens."

print(f"TOKEN\tTOKEN_MATCH\tPREDICTED_LABEL\tTRUE_LABEL\tLABEL_MATCH")
for pred, true in zip(openai_labels, true_labels):
    pred_token, pred_label = pred.split('\t')
    true_token, true_label = true.split('\t')
    
    token_match = pred_token == true_token

    label_match = "✓" if pred_label == true_label else "✗"
    print(f"{pred_token}\t{token_match}\t{pred_label}\t{true_label}\t{label_match}")

TOKEN	TOKEN_MATCH	PREDICTED_LABEL	TRUE_LABEL	LABEL_MATCH
referred	True	O	O	✓
to	True	O	O	✓
as	True	O	O	✓
"	True	O	O	✓
Lender	True	B-PER	B-PER	✓
"),	True	O	O	✓
and	True	O	O	✓
Acme	True	B-ORG	B-ORG	✓
Inc	True	B-ORG	I-ORG	✗
.,	True	O	MISMATCH	✗
a	True	O	MISMATCH	✗
corporation	True	O	MISMATCH	✗
organized	True	O	MISMATCH	✗
and	True	O	MISMATCH	✗
existing	True	O	MISMATCH	✗
under	True	O	MISMATCH	✗
the	True	O	MISMATCH	✗
laws	True	O	MISMATCH	✗
of	True	O	MISMATCH	✗
the	True	O	MISMATCH	✗
United	True	B-LOC	MISMATCH	✗
States	True	I-LOC	MISMATCH	✗
of	True	I-LOC	MISMATCH	✗
America	True	I-LOC	MISMATCH	✗
,	True	O	MISMATCH	✗
with	True	O	MISMATCH	✗
its	True	O	MISMATCH	✗
principal	True	O	MISMATCH	✗
place	True	O	MISMATCH	✗
of	True	O	MISMATCH	✗
business	True	O	MISMATCH	✗
at	True	O	MISMATCH	✗
123	True	O	MISMATCH	✗
Main	True	O	MISMATCH	✗
Street	True	O	MISMATCH	✗
,	True	O	MISMATCH	✗
Anytown	True	B-LOC	GOLD_SKIPPED	✗
,	True	O	O	✓
USA	True	B-LOC	MISMATCH	✗
12345	True	O	MISMATCH	✗
(	True	O	MISMATCH	✗
hereinafter	T

#### Handling nested structure of sentences and tokens to annotate multiple sentences in one prompt

In [ ]:
parsed = parse_iob2_file("FIN5_validation.txt")
from annotate_data import build_chunks

contract_5_sentences = parsed[0]
contract_5_labels = parsed[1]

chunks = build_chunks(contract_5_sentences, contract_5_labels, max_chunk_size=250)

with open("chunks.json", "w") as f:
    json.dump(chunks, f, indent=4, default=lambda x: float(x))

c:\Users\marib\anaconda3\envs\nlp-project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
contract_5_sentences = parsed[0]
contract_5_labels = parsed[1]

chunks

[{'size': 215,
  'tokens': ['Loan',
   'Agreement',
   'This',
   'Loan',
   'Agreement',
   '(',
   'this',
   '“',
   'Agreement',
   '”)',
   'is',
   'made',
   'and',
   'entered',
   'into',
   'by',
   'and',
   'between',
   'the',
   'parties',
   'listed',
   'below',
   'as',
   'of',
   'the',
   '19th',
   'day',
   'of',
   'October',
   ',',
   '2004',
   'in',
   'Beijing',
   ':',
   '(',
   '1',
   ')',
   'Lenovo',
   '-',
   'AsiaInfo',
   'Technologies',
   ',',
   'Inc',
   '.',
   '(“',
   'Lender',
   '”),',
   'a',
   'limited',
   'company',
   'duly',
   'organized',
   'and',
   'existing',
   'under',
   'the',
   'laws',
   'of',
   'the',
   'People',
   '’',
   's',
   'Republic',
   'of',
   'China',
   '(“',
   'PRC',
   '”',
   'or',
   '“',
   'China',
   '”)',
   'with',
   'its',
   'address',
   'at',
   '3',
   '/',
   'F',
   'Zhongdian',
   'Information',
   'Tower',
   ',',
   'No',
   '.',
   '6',
   'Zhongguancun',
   'South',
   'Street',
 

Tokenizing text for annotation

In [33]:
import re
import spacy

nlp = spacy.load("en_core_web_sm")

text = open("../synthetic/contracts_train/c2.txt").read()

# Split on blank lines/new sections first
blocks = re.split(r'\n', text)

sentences = []

for block in blocks:
    # block = block.strip()

    if not block:
        continue

    doc = nlp(block)

    for sent in doc.sents:
        sentences.append(sent.text.strip())

sentences[40:60]

['"Loan Term" means the term of the Loan, which is five (5) years, commencing on June 8, 2025 and ending on June 8, 2030.',
 '"Repayment Terms" means the terms under which the Borrower shall make monthly repayments of the Loan Amount, together with any accrued interest, for a term of five (5) years commencing on August 8, 2025.',
 '"Repayment Schedule" means the schedule attached hereto as Exhibit A, setting forth the dates and amounts of the Borrower\'s monthly repayments.',
 '"Repayment Date" means the date on which each monthly repayment is due.',
 '"Interest" means the amount payable by the Borrower to the Lender in respect of the Loan, calculated at the Interest Rate.',
 '"Fees" means any fees, charges, or expenses payable by the Borrower to the Lender in connection with the Loan.',
 '6. Representations and Warranties',
 '6.1 The Borrower hereby represents and warrants to the Lender that:',
 '6.1.1',
 'The Borrower is a duly organized and existing legal entity under the laws of No

In [34]:
import re

fixed = []

i = 0

while i < len(sentences):

    current = sentences[i]

    # case 1 numbered section like 9.
    if re.match(r'^\d+\.$', current):

        if i + 1 < len(sentences):
            current += " " + sentences[i + 1]
            i += 1


    # case 2: don+t break clause markers like (iv)
    elif re.match(r'^\([a-zA-Z0-9ivx]+\)$', current):

        if i + 1 < len(sentences):
            current += " " + sentences[i + 1]
            i += 1


    # if the sentence starts with parenthesis and it is not a clause marker, it continues the prev sentence and they should go together
    elif (re.match(r'^\(', current)) and not (re.match(r'^\([a-zA-Z0-9ivx]+\)\s+', current)):

        # merge with previous sentence
        if fixed:
            fixed[-1] += " " + current
            i += 1
            continue

    # append final version of current
    fixed.append(current)

    # move to next sentence
    i += 1

In [37]:
def extract_sentences(text):
    nlp = spacy.load("en_core_web_sm")

    # Split on blank lines/new sections first
    blocks = re.split(r'\n', text)

    sentences = []

    for block in blocks:

        if not block:
            continue

        doc = nlp(block)

        for sent in doc.sents:
            sentences.append(sent.text.strip())

    # clean the sentence list
    fixed = []
    i = 0

    while i < len(sentences):

        current = sentences[i]

        # case 1 numbered section like 9.
        if re.match(r'^\d+\.$', current):

            if i + 1 < len(sentences):
                current += " " + sentences[i + 1]
                i += 1


        # case 2: don+t break clause markers like (iv)
        elif re.match(r'^\([a-zA-Z0-9ivx]+\)$', current):

            if i + 1 < len(sentences):
                current += " " + sentences[i + 1]
                i += 1


        # if the sentence starts with parenthesis and it is not a clause marker, it continues the prev sentence and they should go together
        elif (re.match(r'^\(', current)) and not (re.match(r'^\([a-zA-Z0-9ivx]+\)\s+', current)):

            # merge with previous sentence
            if fixed:
                fixed[-1] += " " + current
                i += 1
                continue

        # append final version of current
        fixed.append(current)

        # move to next sentence
        i += 1

    return fixed

In [ ]:
with open("../synthetic/contracts_train/c1.txt", "r") as f:
    text1 = f.read()

tokenized1 = extract_sentences(text1)
tokenized1

['1. Introductory Clause',
 'This Agreement is entered into as of January 1, 2023, by and between Goldman Sachs Bank, USA, a national banking association organized and existing under the laws of the United States of America, with its principal place of business at One New York Plaza, New York, NY 10004, USA (hereinafter referred to as "Lender"), and Acme Inc., a corporation organized and existing under the laws of the United States of America, with its principal place of business at 123 Main Street, Anytown, USA 12345 (hereinafter referred to as "Borrower").',
 'The Borrower is also acting on behalf of its subsidiary, Acme Corp., a corporation organized and existing under the laws of the United States of America, with its principal place of business at 456 Elm Street, Anytown, USA 56789 (hereinafter referred to as "Guarantor").',
 'The Lender is willing to lend to the Borrower, and the Borrower is willing to borrow from the Lender, the sum of Ten Million United States Dollars ($10,000,

In [39]:
tokenized_sentences = []

pattern = r'\w+|[^\w\s]+'

for sent in tokenized1:

    tokens = re.findall(pattern, sent)

    tokenized_sentences.append(tokens)

tokenized_sentences

[['1', '.', 'Introductory', 'Clause'],
 ['This',
  'Agreement',
  'is',
  'entered',
  'into',
  'as',
  'of',
  'January',
  '1',
  ',',
  '2023',
  ',',
  'by',
  'and',
  'between',
  'Goldman',
  'Sachs',
  'Bank',
  ',',
  'USA',
  ',',
  'a',
  'national',
  'banking',
  'association',
  'organized',
  'and',
  'existing',
  'under',
  'the',
  'laws',
  'of',
  'the',
  'United',
  'States',
  'of',
  'America',
  ',',
  'with',
  'its',
  'principal',
  'place',
  'of',
  'business',
  'at',
  'One',
  'New',
  'York',
  'Plaza',
  ',',
  'New',
  'York',
  ',',
  'NY',
  '10004',
  ',',
  'USA',
  '(',
  'hereinafter',
  'referred',
  'to',
  'as',
  '"',
  'Lender',
  '"),',
  'and',
  'Acme',
  'Inc',
  '.,',
  'a',
  'corporation',
  'organized',
  'and',
  'existing',
  'under',
  'the',
  'laws',
  'of',
  'the',
  'United',
  'States',
  'of',
  'America',
  ',',
  'with',
  'its',
  'principal',
  'place',
  'of',
  'business',
  'at',
  '123',
  'Main',
  'Street',
  '